# NB_OTT_GenerateEmbeddings

Objetivo:
Generar una representación vectorial (embedding) para cada ticket de Silver.Tickets utilizando el contenido textual previamente preparado en text_for_embedding.

Este notebook forma parte del pipeline operacional.

IMPORTANTE:
 - incident_id NO se utiliza para generar embeddings.
 - La evaluación semántica se realiza por separado en NB_OTT_EmbeddingEvaluation.

In [1]:
# Parámetro configurable desde Fabric Pipeline.
embedding_model = "all-MiniLM-L6-v2"

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
# ============================================================
# 1. Imports y configuración
#
# Las dependencias se gestionan mediante el Environment ENV_OTT_TicketIntelligence. No se realizan instalaciones dinámicas mediante %pip para mantener la reproducibilidad del pipeline.
# ============================================================

import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer
from pyspark.sql import functions as F


MODEL_NAME = str(embedding_model)

# El modelo se almacena localmente en OneLake para evitar
# depender de una descarga externa durante cada ejecución.
MODEL_PATH = (
    f"/lakehouse/default/Files/Models/{MODEL_NAME}"
)

print("Embedding configuration")
print("-----------------------")
print("Model:", MODEL_NAME)
print("Model path:", MODEL_PATH)

In [ ]:
# ============================================================
# 2. Carga de tickets preparados en Silver
#
# Silver.Tickets ya contiene text_for_embedding, construido previamente a partir del título y descripción normalizados.
#
# Solo seleccionamos las columnas necesarias para esta etapa. incident_id queda deliberadamente fuera de este proceso.
# ============================================================

df_texts = (
    spark.table("Silver.Tickets")
    .select(
        "ticket_id",
        "text_for_embedding"
    )
    .filter(
        F.col("ticket_id").isNotNull()
        &
        F.col("text_for_embedding").isNotNull()
    )
)

source_count = df_texts.count()

print(
    f"Tickets available for embedding: {source_count}"
)

assert source_count > 0, (
    "No tickets available in Silver.Tickets."
)

In [ ]:
# ============================================================
# 3. Carga del modelo Sentence Transformer
#
# all-MiniLM-L6-v2 fue seleccionado para realizar este proyecto por ofrecer un buen equilibrio entre:
#
# - coste computacional,
# - velocidad,
# - calidad semántica,
# - tamaño reducido,
# - embeddings de 384 dimensiones.
#
# Se ejecuta en CPU para mantener la solución reproducible dentro del entorno utilizado para el proyecto.
# ============================================================

model = SentenceTransformer(
    MODEL_PATH,
    device="cpu"
)

embedding_dimension = (
    model.get_sentence_embedding_dimension()
)

print("Model loaded successfully.")
print(
    "Embedding dimension:",
    embedding_dimension
)

In [ ]:
# ============================================================
# 4. Preparación de los datos para inferencia
#
# Para el dataset del prototipo (~1.000 tickets), la conversión a Pandas es suficiente y simplifica la inferencia con SentenceTransformers.
#
# En un escenario de producción con grandes volúmenes, esta etapa debería evolucionar hacia procesamiento distribuido o micro-batches.
# ============================================================

pdf_texts = df_texts.toPandas()

print(
    "Tickets loaded into memory:",
    len(pdf_texts)
)

assert len(pdf_texts) == source_count

In [ ]:
# ============================================================
# 5. Generación de embeddings
#
# Se procesan los tickets en batch en lugar de ejecutar una inferencia individual por ticket. normalize_embeddings=True genera vectores normalizados.
# Esto permite utilizar posteriormente similitud coseno de forma eficiente y es coherente con DBSCAN(metric="cosine").
# ============================================================

embeddings = model.encode(
    pdf_texts[
        "text_for_embedding"
    ].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(
    "Embedding matrix shape:",
    embeddings.shape
)

In [ ]:
# Validación
assert embeddings.shape[0] == source_count, (
    "Number of embeddings differs from number of tickets."
)

assert embeddings.shape[1] == embedding_dimension, (
    "Unexpected embedding dimension."
)

In [ ]:
# ============================================================
# 6. Construcción del dataset de embeddings
#
# Silver.TicketEmbeddings contiene únicamente:
#
# - ticket_id
# - embedding
# - modelo utilizado
# - dimensión
# - timestamp de procesamiento
#
# No duplicamos text_for_embedding porque ya está disponible en Silver.Tickets.
# ============================================================

pdf_embeddings = pd.DataFrame({
    "ticket_id":
        pdf_texts["ticket_id"],

    "embedding":
        [
            vector.tolist()
            for vector in embeddings
        ],

    "embedding_model":
        MODEL_NAME,

    "embedding_dimension":
        int(embedding_dimension)
})

df_embeddings = (
    spark.createDataFrame(
        pdf_embeddings
    )
    .withColumn(
        "_processed_at",
        F.current_timestamp()
    )
)

print(
    "Embeddings prepared:",
    df_embeddings.count()
)

In [ ]:
# ============================================================
# 7. Persistencia en Silver
#
# Los embeddings se almacenan de forma separada de los datos operacionales del ticket.
#
# Esto permite cambiar el modelo de embeddings sin modificar Silver.Tickets y mantiene separadas las responsabilidades de cada tabla.
# ============================================================

(
    df_embeddings.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        "Silver.TicketEmbeddings"
    )
)

print(
    "Silver.TicketEmbeddings written successfully."
)

In [ ]:
# ============================================================
# 8. Validaciones automáticas
#
# Sustituimos las comprobaciones manuales del desarrollo por controles automáticos adecuados para una ejecución orquestada.
# ============================================================

df_saved = spark.table(
    "Silver.TicketEmbeddings"
)

saved_count = df_saved.count()

duplicate_count = (
    df_saved
    .groupBy("ticket_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

invalid_dimension_count = (
    df_saved
    .filter(
        F.col("embedding_dimension")
        != embedding_dimension
    )
    .count()
)

assert saved_count == source_count, (
    f"Expected {source_count} embeddings "
    f"but found {saved_count}."
)

assert duplicate_count == 0, (
    f"Found {duplicate_count} duplicated ticket IDs."
)

assert invalid_dimension_count == 0, (
    "Unexpected embedding dimensions found."
)


print("Embedding quality checks")
print("------------------------")
print("Source tickets:", source_count)
print("Stored embeddings:", saved_count)
print("Duplicate ticket IDs:", duplicate_count)
print(
    "Embedding dimension:",
    embedding_dimension
)

print(
    "\nNB_OTT_GenerateEmbeddings "
    "completed successfully."
)